In [0]:
cus_df = spark.read.table("01_bronze_catalog.raw_schema.customer")
cus_df.show(5)
cus_df.printSchema()

In [0]:
# dropping rows where customer_id null and filling with unknown for string columns
cus_df = cus_df.dropna(subset=["customer_id"])
cus_df = cus_df.fillna("unknown",subset=["customer_name","country","industry_type"])

In [0]:
cus_df = cus_df.withColumn("is_active",col("is_active").cast("boolean")) # changing integer is_active to boolean
cus_df.printSchema()

In [0]:
cus_df = cus_df.fillna("1900-01-01",subset=["account_created_date"])

from pyspark.sql.functions import col
cus_df = cus_df.withColumn("account_created_date",col("account_created_date").cast("date"))

In [0]:
cus_df.printSchema()

In [0]:
emp_df = spark.read.table("01_bronze_catalog.raw_schema.employee")
emp_df.printSchema()

In [0]:
print(emp_df.count()) # 400 rows 
emp_df = emp_df.dropna(subset=["employee_id"])
print(emp_df.count()) # 389 rows

In [0]:
# filling "unknown" for string columns
# filling "1900-01-01" for hire_date column"

emp_df = emp_df.fillna("unknown",subset=["employee_name","role","region"])
emp_df = emp_df.fillna("1900-01-01",subset=["hire_date"])
emp_df.show(5)


In [0]:
# changing is_active_flag to boolean

from pyspark.sql.functions import when,lit;
emp_df = emp_df.withColumn("is_active", when(col("is_active_flag") == "Yes", lit(True)).otherwise(lit(False)))
emp_df = emp_df.drop("is_active_flag")
emp_df.show(5)
emp_df.printSchema()

In [0]:
date_row = emp_df.groupBy("last_update").count().orderBy("last_update",desc=True).first()
date = date_row["last_update"]
print(date)
emp_df = emp_df.fillna(str(date),subset=["last_update"])

In [0]:
emp_df.printSchema()

In [0]:
cus_df.write.mode("overwrite").saveAsTable("02_silver_catalog.transformed_schema.customer")
emp_df.write.mode("overwrite").saveAsTable("02_silver_catalog.transformed_schema.employee")